# 12. The average rating where NULL is not zero

**Track:** sql | **Difficulty:** easy | **Tags:** aggregation, count, avg, nulls, coalesce

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("012-average-rating-nulls")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "reviews": """
UEFSMRUEFbAZFfIMTBWWAxUAEgAA2AwIMXUABQEAMg0IADMNCAA0DQgANQ0IADYNCAA3DQgAOA0IADkNCAA6DQgAOw0IADwNCAA9
DQgAPg0IAD8NCABADQgAQQ0IAEINCABDDQgARA0IAEUNCABGDQgARw0IAEgNCABJDQgASg0IAEsNCABMDQgATQ0IAE4NCABPDQgA
UA0IAFENCABSDQgAUw0IAFQNCABVDQgAVg0IAFcNCABYDQgAWQ0IAFoNCABbDQgAXA0IAF0NCABeDQgAXw0IAGANCABhDQgAYg0I
AGMNCABkDQgAZQ0IAGYNCABnDQgAaA0IAGkNCABqDQgAaw0IAGwNCABtDQgAbg0IAG8NCABwDQgAcQ0IAHINCABzDQgAdA0IAHUN
CAB2DQgAdw0IAHgNCAB5DQgAeg0IAHsNCAB8DQgAfQ0IAH4NCAB/DQgAgA0IAIENCACCDQgAgw0IAIQNCACFDQgAhg0IAIcNCACI
DQgAiQ0IAIoNCACLDQgAjA0IAI0NCACODQgAjw0IAJANCACRDQgAkg0IAJMNCACUDQgAlQ0IAJYNCACXDQgAmA0IAJkNCACaDQgA
mw0IAJwNCACdDQgAng0IAJ8NCACgDQgAoQ0IAKINCACjDQgApA0IAKUNCACmDQgApw0IAKgNCACpDQgAqg0IAKsNCACsDQgArQ0I
AK4NCACvDQgAsA0IALENCACyDQgAsw0IALQNCAC1DQgAtg0IALcNCAC4DQgAuQ0IALoNCAC7DQgAvA0IAL0NCAC+DQgAvw0IAMAN
CADBDQgAwg0IAMMNCADEDQgAxQ0IAMYNCADHDQgAyA0IAMkNCADKDQgAyw0IAMwNCADNDQgAzg0IAM8NCADQDQgA0Q0IANINCADT
DQgA1A0IANUNCADWDQgA1w0IANgNCADZDQgA2g0IANsNCADcDQgA3Q0IAN4NCADfDQgA4A0IAOENCADiDQgA4w0IAOQNCADlDQgA
5g0IAOcNCADoDQgA6Q0IAOoNCADrDQgA7A0IAO0NCADuDQgA7w0IAPANCADxDQgA8g0IAPMNCAD0DQgA9Q0IAPYNCAD3DQgA+A0I
APkNCDz6dQAAAAAAAPt1AAAAAAAAFQAVsgMVugMsFZYDFRAVBhUGHBgI+3UAAAAAAAAYCDF1AAAAAAAAFgAoCPt1AAAAAAAAGAgx
dQAAAAAAABERAAAA2QHw2AMAAACWAwEINQABAgMEBQYHCAkKCwwNDg8QERITFBUWFxgZGhscHR4fICEiIyQlJicoKSorLC0uLzAx
MjM0NTY3ODk6Ozw9Pj9AQUJDREVGR0hJSktMTU5PUFFSU1RVVldYWVpbXF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8
fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaKjpKWmp6ipqqusra6vsLGys7S1tre4ubq7vL2+v8DBwsPExcbH
yMnKAAAAAAAVBBXgAxWKAkwVPBUAEgAA8AEIIgMABQEAJw0IACgNCAAjDQgAJA0IADsNCAA2DQgALA0IADUNCAAzDQgAIQ0IAC8N
CAAyDQgAOA0IADkNCAA8DQgAJg0IACoNCAAxDQgAMA0IADcNCAA9DQgALg0IAC0NCAApDQgAJQ0IADoNCAArDQg8NAMAAAAAAAA+
AwAAAAAAABUAFZYCFZ4CLBWWAxUQFQYVBhwYCD4DAAAAAAAAGAghAwAAAAAAABYAKAg+AwAAAAAAABgIIQMAAAAAAAAREQAAAIsB
8IoDAAAAlgMBBTUgiEGKOSgBtZJgQbR1HCgPogjkJNQwUC0TTEpToqLpkoOBrchIApCoLBML4g35QqBRLjRvk5MhKc64RWg08qbx
YDHheCzJyALEjJXuUN4EtnTJ0XEFsOCGyWGRBMkEkvIgLaIpnG5kgl0i82jPKpvFTLc5dMwrqNaapqSmA1EgAAAAFQQVKBUsTBUK
FQASAAAUTAMAAAACAAAABQAAAAQAAAABAAAAFQAVwgEVyAEsFZYDFRAVBhUGHBgEBQAAABgEAQAAABZiKAQFAAAAGAQBAAAAEREA
AABh8GAfAAAAE7l67/7NUX9vcxABHfbs1/2//j7Nb39/fP3rEAEDBgMpQCIB2hAN2ZQkUwaFWiBCWzSFGIZoQDAJHAYoyyZhxBBo
AAINAiQo06JoGzYIm6QF0yYEGqgIw5QsGAAAFQQZTDUAGAZzY2hlbWEVBgAVBCUCGAlyZXZpZXdfaWQAFQQlAhgKcHJvZHVjdF9p
ZAAVAiUCGAZyYXRpbmcAFpYDGRwZPCYAHBUEGTUABhAZGAlyZXZpZXdfaWQVAhaWAxaIHhbSESacDSYIHBgI+3UAAAAAAAAYCDF1
AAAAAAAAFgAoCPt1AAAAAAAAGAgxdQAAAAAAABERABksFQQVABUCABUAFRAVAgA8KQYZJgCWAwAAACYAHBUEGTUABhAZGApwcm9k
dWN0X2lkFQIWlgMWmgcWzAUmhBQm2hEcGAg+AwAAAAAAABgIIQMAAAAAAAAWACgIPgMAAAAAAAAYCCEDAAAAAAAAEREAGSwVBBUA
FQIAFQAVEBUCADwpBhkmAJYDAAAAJgAcFQIZNQAGEBkYBnJhdGluZxUCFpYDFuoCFvQCJu4XJqYXHBgEBQAAABgEAQAAABZiKAQF
AAAAGAQBAAAAEREAGSwVBBUAFQIAFQAVEBUCADwpBhkmYrQCAAAAFowoFpYDJggWkhoAGRwYDEFSUk9XOnNjaGVtYRjMAi8vLy8v
L0FBQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNBQUlBQUFBQkFBSUFBQUFCQUFBQUFNQUFBQ0FBQUFB
T0FBQUFBUUFBQUNjLy8vL0FBQUJBaEFBQUFBWUFBQUFCQUFBQUFBQUFBQUdBQUFBY21GMGFXNW5BQUNJLy8vL0FBQUFBU0FBQUFE
TS8vLy9BQUFCQWhBQUFBQWNBQUFBQkFBQUFBQUFBQUFLQUFBQWNISnZaSFZqZEY5cFpBQUF2UC8vL3dBQUFBRkFBQUFBRUFBVUFB
Z0FCZ0FIQUF3QUFBQVFBQkFBQUFBQUFBRUNFQUFBQUNRQUFBQUVBQUFBQUFBQUFBa0FBQUJ5WlhacFpYZGZhV1FBQUFBSUFBd0FD
QUFIQUFnQUFBQUFBQUFCUUFBQUFBQUFBQUE9ABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAyNS4wLjEZPBwAABwAABwAAAAd
AwAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "012-average-rating-nulls"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

reviews = spark.read.parquet((_data_dir / "reviews.parquet").resolve().as_posix())
reviews.createOrReplaceTempView("reviews")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'avg_rating': '190e6ceed4b894edd288c1676dd725df85627957933de5d5de464ead7514f533',
                         'product_id': '8a3fa1fde5a340c4658b33821b91bf198a050d2778d8e004987db7cc4c51e490',
                         'rated_reviews': 'ae6a08ff94a73a023b099ed7c1d8d316bb1b6452c6cac3153c2f8d016fcd5b9a',
                         'total_reviews': 'cb972fc617abb3e5d6a10f9ab442c3939780f88f0f4c667e11cbffdaac6a6529'},
 'columns': [['avg_rating', 'double'],
             ['product_id', 'bigint'],
             ['rated_reviews', 'bigint'],
             ['total_reviews', 'bigint']],
 'fingerprint': '8055f5396405a8b3804adcfdee7bf63e8fd6a7e62233327933cdb58d25a63afa',
 'order_matters': False,
 'row_count': 30}

check = make_check(EXPECTED)

## The situation

Customers can leave a written review without giving a star rating. The product page
shows three numbers per product: how many reviews it has, how many of those carry a
rating, and the average rating.

A previous version of this report showed products with good reviews at 2.9 stars.
Somebody had "cleaned" the missing ratings first.

## Input tables

### `reviews`

One row per review.

| Column | Type | Description |
|---|---|---|
| `review_id` | BIGINT | Unique per review. |
| `product_id` | BIGINT | The product being reviewed. |
| `rating` | INT | 1 to 5 stars. NULL means the customer wrote a review without rating the product. |

## Requirement

Return one row per product.

- `total_reviews` is the number of reviews of the product, rated or not.
- `rated_reviews` is the number of reviews that have a rating.
- `avg_rating` is the average of the ratings that exist. A review without a rating
  does not count as zero stars; it does not count at all.
- A product with no rated reviews has `rated_reviews` = 0 and `avg_rating` = NULL.
- Do not round `avg_rating`.

## Expected output

One row per product.

| Column | Type | Description |
|---|---|---|
| `product_id` | BIGINT | |
| `total_reviews` | BIGINT | All reviews. |
| `rated_reviews` | BIGINT | Reviews with a rating. |
| `avg_rating` | DOUBLE | Average rating. NULL if no review is rated. |

In [ ]:
reviews.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `reviews`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)